In [ ]:
import scanpy as sc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import time
import warnings
warnings.filterwarnings("ignore")
import scpositioner as sp
from scpositioner.tl import StabilityAnalysis

In [ ]:
# demo data can be downloaded via https://drive.google.com/drive/folders/1J3IR2S9rOtZWgvZ-Lzp4WYzmNlzPyLi5?usp=drive_link
sc_adata = sc.read('Cerebellum_sc.h5ad')
st_adata = sc.read('Cerebellum_st_n5.h5ad')

In [ ]:
# Similarly, the deconvolution result is required. If you don't have, please see our tutorial to run cell2location.
# If it's not provided, scPositioner will run cell2location automatically.
# Here we provide a c2l result csv for convenience.
deconv_res = pd.read_csv('C2L_cerebellum_n5.csv', index_col=0)
deconv_res

In [ ]:
mapper = StabilityAnalysis(
            S = sc_adata,
            R = st_adata,
            deconv_res = deconv_res, # = None, if it's not provided
            celltype_key = 'CellType',
            estimate_cell_number_list = None,
            mean_cell_numbers = 5,
            normalize = True,
            numItermax = 1e6,
            dropout_rates = [0, 0.1, 0.2, 0.3, 0.5, 0.8, 0.9, 0.92, 0.95, 0.97, 0.98, 0.99],
            seeds = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]) # here to set repeated times, repeated times = length of seeds

In [ ]:
adata_overall_stability = mapper.run()

In [ ]:
# The assigned spot of each cell in each run is stored in obs, named 'dropout_rate_x_seed_y'.
# For each dropout rate, this function calculate a stability score, named 'dropout_rate_x_stability'
# And for all runs, calculate an overall stability score for each cell named 'overall_stability',
# the most frequent assigned spot of this cell is stored as 'final_assigned_spot'.
adata_overall_stability.obs

In [ ]:
adata_overall_stability.write('cb_stability.h5ad')

In [ ]:
adata_overall_stability = sc.read('cb_stability.h5ad')

In [ ]:
# Here we visualize to see the spatial maps of stability scores across different dropout rates
dropout_rates = [0, 0.1, 0.2, 0.3, 0.5, 0.8, 0.9, 0.92, 0.95, 0.97, 0.98, 0.99]

plt.rcParams["figure.figsize"] = (2, 2)
sc.settings.set_figure_params(dpi=300, vector_friendly=True)
sc.pl.embedding(adata_overall_stability, basis='spatial', color=[f'dropout_rate_{dropout_rate}_stability' for dropout_rate in dropout_rates], s=5, vmin=0, vmax=1)

In [ ]:
# And the line plot of stability scores across different dropout rates
sns.set(style="whitegrid")
stability_cols = [f'dropout_rate_{dropout_rate}_stability' for dropout_rate in dropout_rates]

df_stability = adata_overall_stability.obs[stability_cols].copy()
df_stability.columns = [f'DR_{dropout_rate}' for dropout_rate in dropout_rates]

df_stability = df_stability.melt(var_name='dropout_type', value_name='stability')
df_stability['dropout_type'] = pd.Categorical(
    df_stability['dropout_type'],
    categories=[f'DR_{dropout_rate}' for dropout_rate in dropout_rates],
    ordered=True
)

plt.figure(figsize=(4.8, 4.8), dpi=300)

ax = sns.lineplot(
    data=df_stability,
    x="dropout_type",
    y="stability",
    estimator="mean",
    errorbar="sd",
    marker="o",
    linewidth=2,
    markersize=6,
    color="#2f4b7c"
)

ax.set_ylim(0, 1)
ax.set_title("Mapping stability by dropout rate", pad=10)
ax.set_xlabel("Dropout rate")
ax.set_ylabel("Stability score")
plt.xticks(rotation=25, ha="right")

plt.tight_layout()
#plt.savefig('stability_score_lineplot.svg', bbox_inches='tight')
plt.show()

In [ ]:
# Here we analyze stability scores from cell-type level
sc.pl.embedding(adata_overall_stability, basis='spatial', color='CellType', s=5)

sns.set_theme(style="ticks", context="paper", font_scale=1.2)
df_plot = adata_overall_stability.obs[['CellType', 'overall_stability']].dropna()

cell_types = (
    df_plot.groupby('CellType')['overall_stability']
    .median()
    .sort_values(ascending=False)
    .index.tolist()
)

df_plot['CellType'] = pd.Categorical(
    df_plot['CellType'],
    categories=cell_types,
    ordered=True
)

scanpy_celltypes = adata_overall_stability.obs["CellType"].cat.categories
scanpy_colors = adata_overall_stability.uns["CellType_colors"]
celltype_color_dict = dict(
    zip(scanpy_celltypes, scanpy_colors)
)

box_colors = [
    celltype_color_dict[celltype]
    for celltype in cell_types
]

plt.figure(figsize=(10, 6))

ax = sns.boxplot(
    data=df_plot,
    x="CellType",
    y="overall_stability",
    order=cell_types,
    palette=box_colors,
    width=0.7,
    showfliers=False,
    linewidth=1.2,
    boxprops=dict(edgecolor="black"),
    medianprops=dict(color="black", linewidth=1.5),
    whiskerprops=dict(linewidth=1.2),
    capprops=dict(linewidth=1.2)
)

ax.set_xlim(-0.5, len(cell_types)-0.5)
ax.margins(x=0.02)
ax.set_ylim(0, 1.05)
ax.set_title(
    "Mapping stability by cell type",
    pad=10
)
ax.set_xlabel("")
ax.set_ylabel("Overall stability")

plt.xticks(
    rotation=45,
    ha="right"
)
ax.tick_params(
    axis="both",
    which="both",
    direction="in",
    length=5,
    width=1.2,
    color="black"
)

plt.tight_layout()
#plt.savefig('../review/figures/stability_celltype_boxplot_n5.svg', bbox_inches='tight')
plt.show()